In [35]:
import dagshub
import mlflow
from mlflow.tracking import MlflowClient
import joblib
import numpy as np

dagshub.init(repo_owner="adola23", repo_name="HousePricesML", mlflow=True)

# FORCE correct connection
mlflow.set_tracking_uri("https://dagshub.com/adola23/HousePricesML.mlflow")
mlflow.set_registry_uri("https://dagshub.com/adola23/HousePricesML.mlflow")

Initialized MLflow to track repo "adola23/HousePricesML"

Repository adola23/HousePricesML initialized!

# მოდელის და პრეპროცესინგის ჩამოტვირთვა

In [36]:
MODEL_NAME = "bestmodel2"
MODEL_VERSION = 2

model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/{MODEL_VERSION}")

In [37]:

client = MlflowClient()

model_version = client.get_model_version(
    name=MODEL_NAME,
    version=MODEL_VERSION
)

run_id = model_version.run_id

from mlflow.tracking import MlflowClient

artifacts = client.list_artifacts(run_id)

for a in artifacts:
    print(a.path)

correlation_dropped.json
preprocessing.pkl


In [38]:
preprocessing_path = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path="preprocessing.pkl"
)

artifacts = joblib.load(preprocessing_path)

In [39]:
num_cols = artifacts["num_cols"]
cat_cols = artifacts["cat_cols"]
train_means = artifacts["train_means"]
one_hot_used_cols = artifacts["one_hot_used_cols"]
one_hot_columns = artifacts["one_hot_columns"]
selected_cols = artifacts["selected_cols"]
to_drop = artifacts["to_drop"]

In [40]:
import pandas as pd

test = pd.read_csv("data/test.csv")

# პრეპროცესინგი

In [41]:
X = test.drop(columns=["Id"])

In [42]:
for col in num_cols:
    X[col].fillna(train_means[col], inplace=True)

for col in cat_cols:
    if col in one_hot_used_cols:
        dummies = pd.get_dummies(X[col], prefix=col)
        dummies = dummies.reindex(columns=one_hot_columns[col], fill_value=0)

        X = pd.concat([X.drop(columns=[col]), dummies], axis=1)
    else:
        X.drop(columns=[col], inplace=True)

X = X.reindex(columns=selected_cols, fill_value=0)
X.drop(columns=to_drop, inplace=True)

# პასუხების მიღება

In [43]:
Y_log = model.predict(X)
Y = np.expm1(Y_log)
print(Y)

[124398.81518828 140732.68513319 177522.77800777 ... 170781.32188287
 111054.56169101 233616.73770823]


In [44]:
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": Y
})

submission.to_csv("submission.csv", index=False)